# Low-Mass Cooling-Flow Example

This notebook uses the modified-Plummer halo and float-native diagnostics for a lower-mass system.


In [ ]:
import sys
from pathlib import Path

def _find_repo_root():
    for start in (Path.cwd(), Path.cwd().resolve()):
        for candidate in (start, *start.parents):
            if (candidate / 'pysrc' / 'solve_ode.py').exists():
                return candidate
    raise RuntimeError('Could not find repo root containing pysrc/solve_ode.py')

repo_root = _find_repo_root()
pysrc_path = repo_root / 'pysrc'
if str(pysrc_path) not in sys.path:
    sys.path.insert(0, str(pysrc_path))

import numpy as np

import solve_ode as CF
import HaloPotential_new as Halo
import WiersmaCooling as Cool
from analytic_models import RotationConfig, TurbulenceConfig
from cosmology import DEFAULT_COSMOLOGY


In [ ]:
rho_mean = DEFAULT_COSMOLOGY.mean_matter_density_Msun_kpc3(0.0)
potential = Halo.CombinedPotential_using_modified_plummer(
    M_vir_Msun=8.0e10,
    r_vir_kpc=100.0,
    c_vir=11.0,
    M_gal_Msun=8.0e9,
    a_gal_kpc=2.5,
    b_gal_kpc=0.35,
    rho_mean_Msun_kpc3=rho_mean,
    R200_kpc=130.0,
)
cooling = Cool.Constant_Cooling(1.0e-22)
solution = CF.IntegrateFlowEquations(
    mass_flow_rate_Msun_per_yr=0.01,
    temperature_K=3.0e5,
    density_cgs=1.2e-27,
    potential=potential,
    cooling=cooling,
    direction=1,
    R_min_kpc=20.0,
    R_max_kpc=70.0,
)
rotation = solution.rotation_diagnostics(RotationConfig(R_circ_kpc=8.0))
turbulence = solution.turbulence_diagnostics(TurbulenceConfig())
{
    'disk_interface_radius_kpc': float(rotation.disk_interface_radius_kpc),
    'max_sigma_turb_kms': float(np.max(turbulence.sigma_turb_kms)),
}
